## Analysis of the PREACT-digital study sample

In [ ]:
from pathlib import Path
import sys
from datetime import date
import pandas as pd
import gc
import os
import glob
import numpy as np
import pickle

# --- Paths / imports -------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

from server_config import (
    datapath,
    proj_sheet,
    preprocessed_path, redcap_path,
    raw_path,
    backup_path,
)

In [ ]:
with open(preprocessed_path + '/monitoring_data.pkl', 'rb') as file:
    df_monitoring = pickle.load(file)
    
with open(redcap_path + '/redcap_data.pkl', 'rb') as file:
    df_redcap = pickle.load(file)
df_redcap_final = pd.read_spss(redcap_path + "/sp1_cleaned" + "/baseline_raw_20251012.sav")

split_path = redcap_path + "/sp1_cleaned" +  "/111025_subject_splits.csv"
df_split = pd.read_csv(split_path)

### Check study timeframes

In [ ]:

t5_path_zert = redcap_path + "/t5" + "/Zertifizierung_FOR_Verlaufsdoku_T5.csv"
t5_path = redcap_path + "/t5" + "/FOR_Verlaufsdoku_T5.csv"

t5_zert = pd.read_csv(t5_path_zert, low_memory="False")
t5 = pd.read_csv(t5_path, low_memory="False")

In [ ]:
cols_to_keep = ['for_id','t5_session_date', 't1_session_date','t20_session_date','t5_assessment_date','t20_assessment_date','post_assessment_date', 'ic_date']

In [ ]:
t5_short = t5[cols_to_keep]
t5_zert_short = t5_zert[cols_to_keep]
t5_final = pd.concat([t5_zert_short, t5_short], ignore_index=True)

In [ ]:
t5_final = t5_final.drop_duplicates()

In [ ]:
# (optional) ensure it's a datetime so sorting behaves correctly
t5_final['t5_assessment_date'] = pd.to_datetime(t5_final['t5_assessment_date'], errors='coerce')

dedup = (
    t5_final.sort_values(['for_id', 't5_assessment_date'], na_position='first')  # NaNs first
      .drop_duplicates(subset='for_id', keep='last')                       # keep latest non-null
      .reset_index(drop=True)
)


### Check baseline demographics 

In [ ]:
df_redcap_final["ema_participation"] = (
    df_redcap_final["ema_start_date"].notna().map({True: "yes", False: "no"})
)

In [ ]:
df_redcap_final["ema_participation"].value_counts(dropna=False)